In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F
from torch.utils.data import random_split
from sklearn.metrics import f1_score
from tqdm import tqdm


In [3]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 4 * 4, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 4 * 4)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [4]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
full_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)

# Split the dataset into train, validation, and test sets
train_size = int(0.9 * len(full_dataset))
val_size = int(0.1 * len(full_dataset))
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)



In [5]:
batch_size = 32
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=True)


In [6]:
def calc_metrics(loader, model):
    correct = 0
    total = 0
    all_labels = []
    all_predictions = []
    with torch.no_grad():
        for data in loader:
            images, labels = data
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            all_labels.extend(labels.numpy())
            all_predictions.extend(predicted.numpy())
    accuracy = 100 * correct / total
    f1 = f1_score(all_labels, all_predictions, average='macro')
    return accuracy, f1

In [9]:
store_loss = []
store_F1 = []
store_acc = []
net = Net()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)
# Train the network
for epoch in range(10):
    print("-"*100)

    running_loss = 0.0
    cnt = 0
    for i, data in tqdm(enumerate(train_loader, 0)):
        inputs, labels = data
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if i % 100 == 99:
            val_accuracy, val_f1 = calc_metrics(val_loader, net)
            print(f"{epoch + 1}:{i} Validation accuracy: {val_accuracy}, F1: {val_f1}")
            store_F1.append(val_f1)
            store_acc.append(val_accuracy)
            torch.save(net.state_dict(), f'models/{epoch}-{cnt}-CNN.pt')
            cnt += 1
            if val_accuracy > 95:
                break
    if val_accuracy > 95:
        break

    print(f'Epoch {epoch + 1}, loss: {running_loss / len(train_loader)}')
    optimizer.zero_grad()
    store_loss.append(running_loss)
    
    
print('Finished Training')

----------------------------------------------------------------------------------------------------


109it [00:04,  9.55it/s]

1:99 Validation accuracy: 8.9, F1: 0.02296428441619665


205it [00:08,  9.91it/s]

1:199 Validation accuracy: 10.1, F1: 0.02230862502308625


...

2:899 Validation accuracy: 94.53333333333333, F1: 0.9446828131775267


1004it [01:08,  1.47it/s]

2:999 Validation accuracy: 94.56666666666666, F1: 0.9450528130870784


1099it [01:14, 14.82it/s]

2:1099 Validation accuracy: 95.01666666666667, F1: 0.9496260749254153
Finished Training


In [10]:
print(net.state_dict()["fc1.weight"])
print(net.state_dict()["fc2.weight"])
print(net.state_dict()["fc3.weight"])


tensor([[ 0.0505,  0.0143,  0.0395,  ..., -0.0441, -0.0050, -0.0312],
        [ 0.0539, -0.0372,  0.0166,  ..., -0.0376, -0.0406, -0.0171],
        [ 0.0413, -0.0505, -0.0367,  ..., -0.0795, -0.0202, -0.0167],
        ...,
        [ 0.0304,  0.0319,  0.0427,  ...,  0.0405,  0.0169, -0.0550],
        [ 0.0225,  0.0198,  0.0476,  ...,  0.0599, -0.0517,  0.0397],
        [ 0.0195, -0.0046,  0.0150,  ...,  0.0026, -0.0372, -0.0210]])
tensor([[-0.0507, -0.0625,  0.0342,  ..., -0.0702,  0.0737, -0.0035],
        [-0.0171,  0.1151,  0.0397,  ...,  0.0984, -0.0422, -0.0312],
        [ 0.0017,  0.0849, -0.0891,  ...,  0.0612,  0.0040, -0.0334],
        ...,
        [-0.0135,  0.0520,  0.0798,  ...,  0.0446,  0.0170, -0.0856],
        [-0.0466, -0.0224,  0.0810,  ..., -0.0793, -0.0482, -0.0515],
        [ 0.0584,  0.0208, -0.0685,  ..., -0.0772,  0.0143, -0.0087]])
tensor([[-9.4305e-02, -2.7030e-02, -4.3728e-02,  6.6356e-02,  6.4069e-02,
         -4.0694e-02, -5.5149e-02,  1.0899e-01,  6.7559e-0